# ogbn-proteins: GNN Training with PyTorch Geometric (PyG)

This notebook trains a GNN on the **ogbn-proteins** benchmark from the Open Graph Benchmark (OGB)
using **PyTorch Geometric (PyG)**.

The model supports the following MPNN types:
- `sage` – GraphSAGE
- `gcn`  – Graph Convolutional Network
- `saint` 
- `SGCN` 



## 1. Install Dependencies

Run the cell below **once** if the required packages are not yet installed.

In [1]:
# Uncomment and run if packages are missing
# !pip install torch
# !pip install torch_geometric
# !pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__)')+cu121.html
# !pip install ogb

## 2. Imports

In [2]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models import GNN_PyG
from src.utils import set_seed, load_data, preprocess, gen_model, add_labels
from src.train import train_epoch, evaluate, run
from src.logging_utils import (
    setup_dirs, setup_experiment_dir, save_config, update_experiment_index,
    build_epoch_df, build_run_record,
    save_epoch_metrics, save_run_summary, save_aggregate_summary, compute_aggregate,
)
from src.visualization import (
    plot_loss_curve, plot_auc_curve, plot_auc_boxplot, plot_time_bar,
)

print('All imports successful.')


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


## 3. Configuration

Edit the variables below to configure the experiment (replaces command-line arguments).

In [3]:
# ── Device ──────────────────────────────────────────────────────────────────
USE_CPU  = False   # Set True to force CPU mode
GPU_ID   = 0       # GPU device ID (ignored when USE_CPU=True)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED     = 0
N_RUNS   = 1       # Number of independent runs

# ── Model ────────────────────────────────────────────────────────────────────
MPNN       = 'gcn' # 'sage' | 'gcn' | 'graphsaint' | 'sgcn'
N_LAYERS   = 3
N_HIDDEN   = 64
USE_LABELS = False # Concatenate training labels as input features
JK         = False # Enable Jumping Knowledge (JK) aggregation

# ── SGCN (only used when MPNN = 'sgcn') ────────────────────────────────────
N_SUBGRAPHS        = 5              # number of independent subgraphs per epoch
SUBSAMPLING_METHOD = 'random_node'  # 'random_node' | 'random_edge' | 'random_walk' | 'snowball'
SUBGRAPH_MAX_NODES = 256            # hard upper bound on nodes per subgraph (<= 0: use SUBGRAPH_RATIO)
MAX_SUBGRAPH_EDGES = 300000         # hard upper bound on edges per subgraph (<= 0: no cap)
TRUNCATION_RATIO   = 0.2            # fraction of worst subgraphs to discard (0.0 = keep all)
AGGREGATION_METHOD = 'sgcn'         # 'sgcn' (softmax) | 'avg' (SGCN-Avg) | 'weighted' (SGCN-Weighted)
LOCAL_EPOCHS       = 5              # number of local gradient steps per subgraph (L)
DEBUG_SUBGRAPH_STATS = False        # print per-subgraph shape/memory stats for OOM debugging

# ── Regularisation ───────────────────────────────────────────────────────────
DROPOUT    = 0.25
INPUT_DROP = 0.1
EDGE_DROP  = 0.1

# ── Optimiser ────────────────────────────────────────────────────────────────
LR           = 0.01
WEIGHT_DECAY = 0.0

# ── Training schedule ────────────────────────────────────────────────────────
N_EPOCHS   = 500
EVAL_EVERY = 5
LOG_EVERY  = 5

# ── Misc ─────────────────────────────────────────────────────────────────────
SAVE_PRED  = False  # Save final predictions to ./output/

# ── Dataset constants (do not change) ────────────────────────────────────────
DATASET_NAME = 'ogbn-proteins'
N_NODE_FEATS = 0    # will be set after preprocessing
N_CLASSES    = 112

# ── Device setup ─────────────────────────────────────────────────────────────
if USE_CPU or not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device(f'cuda:{GPU_ID}')

print(f'Using device: {device}')


Using device: cuda:0


## 4. Model Definition

GNN_PyG supports GraphSAGE and GCN.

In [4]:
# GNN_PyG is imported from src.models
print('Model class imported from src.models.')


Model class defined.


## 5. Utility Functions

In [5]:
# set_seed, load_data, preprocess, gen_model, add_labels are imported from src.utils
print('Utility functions imported from src.utils.')


Utility functions defined.


## 6. Training and Evaluation

In [6]:
# train_epoch, evaluate are imported from src.train
print('Training/evaluation functions imported from src.train.')


Training/evaluation functions defined.


## 7. Main Run Function

In [7]:
# run is imported from src.train
print('Run function imported from src.train.')


Run function defined.


## 8. Load and Preprocess Data

In [8]:
print('Loading data ...')
data, train_idx, val_idx, test_idx, evaluator = load_data(DATASET_NAME)

print('Preprocessing ...')
data = preprocess(data, train_idx, N_CLASSES)
N_NODE_FEATS = data.x.shape[-1]

labels = data.y
labels, train_idx, val_idx, test_idx = (
    labels.to(device),
    train_idx.to(device),
    val_idx.to(device),
    test_idx.to(device),
)

print(f'Node features:  {data.x.shape}')
print(f'Labels:         {labels.shape}')
print(f'Train nodes:    {len(train_idx)}')
print(f'Val nodes:      {len(val_idx)}')
print(f'Test nodes:     {len(test_idx)}')
print(f'N_NODE_FEATS (after preprocess): {N_NODE_FEATS}')


Loading data ...


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.l

Preprocessing ...
Node features:  torch.Size([132534, 8])
Labels:         torch.Size([132534, 112])
Train nodes:    86619
Val nodes:      21236
Test nodes:     24679
N_NODE_FEATS (after preprocess): 8


## 9. Train the Model

In [9]:
all_epoch_dfs  = []
all_run_records = []

for i in range(N_RUNS):
    print(f'\n=== Run {i + 1} / {N_RUNS} ===')
    seed = SEED + i
    set_seed(seed)
    _gen_model = lambda: gen_model(
        N_NODE_FEATS, N_CLASSES, USE_LABELS, N_LAYERS, N_HIDDEN,
        DROPOUT, INPUT_DROP, EDGE_DROP, MPNN, JK
    )
    result = run(
        data, labels, train_idx, val_idx, test_idx, evaluator, n_running=i + 1,
        gen_model_fn=_gen_model, device=device,
        n_layers=N_LAYERS, lr=LR, weight_decay=WEIGHT_DECAY,
        n_epochs=N_EPOCHS, eval_every=EVAL_EVERY, log_every=LOG_EVERY,
        save_pred=SAVE_PRED, use_labels=USE_LABELS, n_classes=N_CLASSES,
        mpnn=MPNN,
        subsampling_method=SUBSAMPLING_METHOD,
        truncation_ratio=TRUNCATION_RATIO,
        aggregation_method=AGGREGATION_METHOD,
        n_subgraphs=N_SUBGRAPHS,
        subgraph_max_nodes=SUBGRAPH_MAX_NODES,
        max_subgraph_edges=MAX_SUBGRAPH_EDGES,
        local_epochs=LOCAL_EPOCHS,
        debug_subgraph_stats=DEBUG_SUBGRAPH_STATS,
    )

    all_epoch_dfs.append(build_epoch_df(MPNN, i + 1, seed, result['epoch_records']))
    all_run_records.append(build_run_record(MPNN, i + 1, seed, result))

    print(
        f'Run {i + 1} finished – '
        f'Val ROC-AUC: {result["best_val_auc"]:.4f} | '
        f'Test ROC-AUC: {result["best_test_auc"]:.4f} | '
        f'Total time: {result["total_run_time"]:.1f}s'
    )

all_val_scores  = [r['best_val_auc']  for r in all_run_records]
all_test_scores = [r['best_test_auc'] for r in all_run_records]
print('\n=== Summary ===')
print(f'Val  ROC-AUC: {np.mean(all_val_scores):.4f} ± {np.std(all_val_scores):.4f}')
print(f'Test ROC-AUC: {np.mean(all_test_scores):.4f} ± {np.std(all_test_scores):.4f}')



=== Run 1 / 1 ===


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch: 0005 | Loss: 0.3169 | Train: 73.40% | Valid: 68.06% | Test: 59.73% | Best Valid: 68.06% | Best Test: 59.73%
Epoch: 0010 | Loss: 0.3069 | Train: 75.90% | Valid: 71.01% | Test: 64.64% | Best Valid: 71.01% | Best Test: 64.64%
Epoch: 0015 | Loss: 0.3002 | Train: 78.17% | Valid: 72.35% | Test: 64.50% | Best Valid: 72.35% | Best Test: 64.50%
Epoch: 0020 | Loss: 0.2848 | Train: 80.81% | Valid: 76.98% | Test: 67.65% | Best Valid: 76.98% | Best Test: 67.65%
Epoch: 0025 | Loss: 0.2737 | Train: 83.88% | Valid: 79.52% | Test: 73.42% | Best Valid: 79.52% | Best Test: 73.42%
Epoch: 0030 | Loss: 0.2689 | Train: 84.37% | Valid: 80.03% | Test: 72.99% | Best Valid: 80.03% | Best Test: 72.99%
Epoch: 0035 | Loss: 0.2662 | Train: 84.91% | Valid: 80.46% | Test: 74.43% | Best Valid: 80.46% | Best Test: 74.43%
Epoch: 0040 | Loss: 0.2639 | Train: 85.16% | Valid: 80.29% | Test: 72.71% | Best Valid: 80.46% | Best Test: 74.43%
Epoch: 0045 | Loss: 0.2627 | Train: 84.90% | Valid: 80.52% | Test: 74.84% | Best

## 10. Save Results to CSV


In [ ]:
config = {
    'method':          MPNN,
    'dataset':         DATASET_NAME,
    'sampling_method': SUBSAMPLING_METHOD,
    'trunc_ratio':     TRUNCATION_RATIO,
    'local_epochs':    LOCAL_EPOCHS,
    'epochs':          N_EPOCHS,
    'lr':              LR,
    'hidden_dim':      N_HIDDEN,
    'seeds':           [SEED + i for i in range(N_RUNS)],
}

exp_dir, figures_dir = setup_experiment_dir('results')
save_config(config, exp_dir, device=device)

epoch_df = save_epoch_metrics(all_epoch_dfs, exp_dir)
run_df   = save_run_summary(all_run_records, exp_dir)
agg_df   = save_aggregate_summary(all_run_records, exp_dir)

update_experiment_index(exp_dir, config, compute_aggregate(all_run_records))

print('\naggregate_summary:')
print(agg_df.to_string(index=False))


## 11. Generate Figures


In [ ]:
plot_loss_curve(epoch_df, figures_dir, method=MPNN)
plot_auc_curve(epoch_df, figures_dir, method=MPNN)
plot_auc_boxplot(run_df, figures_dir, method=MPNN)
plot_time_bar(compute_aggregate(all_run_records), figures_dir, method=MPNN)

print('All figures saved to', figures_dir)
